# AAF to Neptune Graph

Comprehensive parser - filters nodes/edges with N/A IDs

In [1]:
import gzip, json
from pathlib import Path
from typing import Any, Dict, List, Tuple, Union
from boto3 import Session
from neptune_graph_manager.types import Edge, Node, NodeArray
from neptune_graph_manager import GraphBuilder, NeptuneGraphManager
from dotenv import load_dotenv
import os
from ast import literal_eval

load_dotenv()

/Users/crisleoo/workplace/ThreatForest-internal/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
session = Session(**literal_eval(os.getenv("SESSION_PARAMS", {})))
neptune_manager = NeptuneGraphManager(session=session, graph_id="g-f7i4wf2pc5")
summary = neptune_manager.get_summary()

with open('data/graph_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Graph summary saved to data/graph_summary.json")

📈 Graph summary generated successfully!
Graph summary saved to data/graph_summary.json


In [7]:
query = """
MATCH p=()-[:subtechnique-of]-()
RETURN p
"""
neptune_manager.query_ops.execute_query(query=query, visualize=True, interactive=True, max_nodes=100)

Error executing query: An error occurred (ValidationException) when calling the ExecuteQuery operation: Invalid input '-': expected an identifier character, whitespace, "::", '|', a length specification, a property map or ']' (line 2, column 26 (offset: 26))


ValidationException: An error occurred (ValidationException) when calling the ExecuteQuery operation: Invalid input '-': expected an identifier character, whitespace, "::", '|', a length specification, a property map or ']' (line 2, column 26 (offset: 26))

In [5]:
query_embedding = neptune_manager.embedding_ops.get_embedding("Perform an attack to avoid possible detection of tools and activities")

neptune_query = f"""
CALL neptune.algo.vectors.topKByEmbedding(
  {query_embedding}
)
YIELD node, score
RETURN node, score
"""

results = neptune_manager.query_ops.execute_query(neptune_query)
node_results = NodeArray().from_neptune_nodes(results)

for n in node_results:
    print(f"{n.properties['name']} : {n.properties['description'][:200]}")

Disable or Modify Tools : Adversaries may disable security tools to avoid possible detection of their tools and activities. This can take the form of killing security software or event logging processes, deleting Registry keys
Security Software Discovery : Adversaries may attempt to get a listing of security software, configurations, defensive tools, and sensors that are installed on a system or in a cloud environment. This may include things such as fi
Automated Collection : Once established within a system or network, an adversary may use automated techniques for collecting internal data. Methods for performing this technique could include use of a [Command and Scripting
Account Access Removal : Adversaries may interrupt availability of system and network resources by inhibiting access to accounts utilized by legitimate users. Accounts may be deleted, locked, or manipulated (ex: changed crede
Exploitation for Defense Evasion : Adversaries may exploit a system or application vulnerabi

In [6]:
node_results[0]

Node(id=NodeID(id='attack-pattern--a307c339-fd4f-448a-846b-88a98a5fdf79', label='None'), label='Technique', properties={'stix_id': 'attack-pattern--a307c339-fd4f-448a-846b-88a98a5fdf79', 'created': '2020-02-21T20:32:20.810Z', 'description': 'Adversaries may disable security tools to avoid possible detection of their tools and activities. This can take the form of killing security software or event logging processes, deleting Registry keys so that tools do not start at run time, or other methods to interfere with security tools scanning or reporting information.', 'stix_type': 'attack-pattern', 'modified': '2025-01-29T21:26:09.156813Z', 'urls': 'https://attack.mitre.org/techniques/T1562/001,https://capec.mitre.org/data/definitions/578.html,https://w.amazon.com/bin/view/AWS/Teams/GlobalServicesSecurity/TDIR/TRIAD/AAF/Techniques/T1562/001', 'tactics': 'defense-evasion', 'name': 'Disable or Modify Tools', 'external_ids': 'T1562.001,CAPEC-578,T1562.001', 'aliases': 'T1562.001'}, score=152.3